<a href="https://colab.research.google.com/github/harrypotes10/Hoja_de_vida/blob/main/Laboratorio_IA_2026_69_002_Naive_Bayes_Cafe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Laboratorio 2: Fundamentos de Probabilidad, Estadística y Redes Bayesianas
**Asignatura:** Inteligencia Artificial  
**Unidad 2:** Teoría Bayesiana  
**Resultado de Aprendizaje:** Realizar la clasificación de patrones basado en la Teoría Bayesiana (en Python).  
**Dataset de Referencia:** `PRODUCCIONc.csv` (Producción de Café en Colombia)  
**Nivel de Dificultad:** Medio  
**Formato:** Jupyter Notebook (`.ipynb`)  

---

## 📌 Contexto del Problema
La producción agrícola de café en Colombia depende de múltiples factores regionales y físicos, tales como el departamento, la superficie sembrada (`Area (ha)`) y el volumen cosechado (`Produccion (ton)`). El rendimiento de la cosecha (`ha/ton` o `ton/ha`) determina la eficiencia productiva de cada región.

En este laboratorio práctico, aplicará los conceptos fundamentales de la **Teoría Bayesiana** y las **Redes Bayesianas** para construir un sistema capaz de **clasificar patrones de rendimiento cafetero** a partir de datos empíricos.

---

## 📋 Estructura de Trabajo
1. **Preprocesamiento y Exploración de Datos:** Limpieza de formatos numéricos y discretización de variables.
2. **Fundamentos de Probabilidad y Teorema de Bayes:** Cálculo explícito de probabilidades *a priori*, verosimilitudes y probabilidades *posteriori*.
3. **Clasificador Supervisado Naive Bayes:** Entrenamiento y predicción con `GaussianNB` y `CategoricalNB`.
4. **Modelado con Redes Bayesianas:** Definición del Grafo Dirigido Acíclico (DAG) y construcción de Tablas de Probabilidad Condicional (CPT).
5. **Evaluación de Desempeño:** Matriz de confusión, sensibilidad (TPR), especificidad (TNR) y exactitud.
6. **Cuestionario y Trabajo Autónomo:** Preguntas conceptuales y análisis crítico.

---



## 1. Carga, Limpieza y Exploración de Datos

### Tarea 1.1: Carga del dataset y formateo numérico
El dataset `PRODUCCIONc.csv` contiene información histórica sobre el café en Colombia. Las columnas de área y producción contienen comas como separadores de miles y deben ser convertidas a tipo flotante (`float`).

* Complete la función de carga de datos e inspeccione las dimensiones y los tipos de datos.



In [ ]:
# Importación de librerías requeridas
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Carga del dataset (manejo de ruta local o de entorno)
path = 'PRODUCCIONc.csv' if os.path.exists('PRODUCCIONc.csv') else '/workspace/knowledge/PRODUCCIONc.csv'
df = pd.read_csv(path)

# Limpieza de cadenas numéricas con comas
for col in ['Area (ha)', 'Produccion (ton)']:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(',', '').str.replace('"', '').astype(float)

print("✅ Dataset cargado correctamente. Dimensiones:", df.shape)
df.head(5)



### Tarea 1.2: Definición de la Variable Objetivo y Discretización
Para la clasificación de patrones, crearemos una variable categórica objetivo `Rendimiento_Nivel`:
* **Alto:** Rendimiento $\ge 1.10$
* **Medio:** $0.75 \le 	ext{Rendimiento} < 1.10$
* **Bajo:** Rendimiento $< 0.75$

Asimismo, discretizaremos el `Area (ha)` en tres niveles (`Pequena`, `Mediana`, `Grande`) mediante terciles.



In [ ]:
# Creación de la etiqueta de clase objetivo
def categorizar_rendimiento(val):
    if val >= 1.10:
        return 'Alto'
    elif val >= 0.75:
        return 'Medio'
    else:
        return 'Bajo'

df['Rendimiento_Nivel'] = df['Rendimiento (ha/ton)'].apply(categorizar_rendimiento)

# Discretización del área cultivada en 3 categorías
df['Area_Nivel'] = pd.qcut(df['Area (ha)'], q=3, labels=['Pequena', 'Mediana', 'Grande'])

print("Distribución de Clases (Rendimiento_Nivel):")
print(df['Rendimiento_Nivel'].value_counts(normalize=True))

# Muestra de datos transformados
df[['Departamento', 'Area (ha)', 'Area_Nivel', 'Rendimiento (ha/ton)', 'Rendimiento_Nivel']].head(8)



## 2. Aplicación Directa del Teorema de Bayes

La Regla de Bayes establece:

$$P(C_k | X) = rac{P(X | C_k) \cdot P(C_k)}{P(X)}$$

Donde:
* $P(C_k)$ es la **probabilidad a priori** de la clase $C_k$.
* $P(X | C_k)$ es la **verosimilitud** de la evidencia $X$ dada la clase $C_k$.
* $P(X)$ es la **probabilidad marginal (evidencia)**.
* $P(C_k | X)$ es la **probabilidad a posteriori**.

### Tarea 2.1: Cálculo Programático de Probabilidades
Calculemos la probabilidad posterior de que un cultivo tenga **Rendimiento Alto** dado que su superficie es **Grande** ($X = 	ext{'Area\_Nivel: Grande'}$).



In [ ]:
# 1. Probabilidad a priori de la clase P(Rendimiento = Alto)
p_prior_alto = (df['Rendimiento_Nivel'] == 'Alto').mean()

# 2. Verosimilitud P(Area = Grande | Rendimiento = Alto)
df_alto = df[df['Rendimiento_Nivel'] == 'Alto']
p_verosimilitud = (df_alto['Area_Nivel'] == 'Grande').mean()

# 3. Probabilidad marginal de la evidencia P(Area = Grande)
p_evidencia = (df['Area_Nivel'] == 'Grande').mean()

# 4. Probabilidad a posteriori mediante Teorema de Bayes
p_posterior_alto = (p_verosimilitud * p_prior_alto) / p_evidencia

print(f"P(Rendimiento = Alto) [Prior]: {p_prior_alto:.4f}")
print(f"P(Area = Grande | Rendimiento = Alto) [Likelihood]: {p_verosimilitud:.4f}")
print(f"P(Area = Grande) [Evidencia]: {p_evidencia:.4f}")
print(f"--------------------------------------------------")
print(f"P(Rendimiento = Alto | Area = Grande) [Posterior]: {p_posterior_alto:.4f}")



## 3. Clasificación Supervisada con Naive Bayes

El algoritmo Naive Bayes asume independencia condicional entre las características predictoras dado el estado de la clase:

$$P(C_k | x_1, x_2, \dots, x_n) \propto P(C_k) \prod_{i=1}^{n} P(x_i | C_k)$$

### Tarea 3.1: Entrenamiento del Modelo GaussianNB
* Seleccione como predictores continuos: `Area (ha)`, `Produccion (ton)`, `Produccion Nacional (ton)`, y `Area Nacional (ha)`.
* Divida los datos en entrenamiento (80%) y prueba (20%) manteniendo la estratificación de clases.
* Entrene el modelo `GaussianNB` y obtenga el reporte de clasificación.



In [ ]:
# Selección de atributos de entrada (X) y etiqueta de salida (y)
X = df[['Area (ha)', 'Produccion (ton)', 'Produccion Nacional (ton)', 'Area Nacional (ha)']]
y = df['Rendimiento_Nivel']

# División Train/Test estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Inicializar y entrenar el clasificador GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# Predicción en el conjunto de test
y_pred = gnb.predict(X_test)

# Resultados de evaluación
print("Exactitud (Accuracy): {:.2f}%".format(accuracy_score(y_test, y_pred) * 100))
print("
Reporte de Clasificación Supervisada:")
print(classification_report(y_test, y_pred))



## 4. Representación y Factorización de una Red Bayesiana (DAG)

Una **Red Bayesiana** representa un dominio mediante un Grafo Dirigido Acíclico (DAG) y un conjunto de Tablas de Probabilidad Condicional (CPT).

### Estructura Causal Propuesta:
```
       [Departamento]
             │
             ▼
       [Area_Nivel] ────────► [Produccion_Nivel]
             │                      │
             └──────► [Rendimiento_Nivel] ◄──┘
```

La **Distribución de Probabilidad Conjunta** factorizada de la red es:

$$P(D, A, P, R) = P(D) \cdot P(A | D) \cdot P(P | A, D) \cdot P(R | P, A)$$

### Tarea 4.1: Construcción empírica de CPTs
Genere la Tabla de Probabilidad Condicional $P(	ext{Rendimiento\_Nivel} | 	ext{Produccion\_Nivel}, 	ext{Area\_Nivel})$ a partir del conjunto de datos.



In [ ]:
# Discretizar la Producción en 3 niveles
df['Produccion_Nivel'] = pd.qcut(df['Produccion (ton)'], q=3, labels=['Baja', 'Media', 'Alta'])

# Frecuencias cruzadas entre Produccion_Nivel, Area_Nivel y Rendimiento_Nivel
cpt = df.groupby(['Produccion_Nivel', 'Area_Nivel', 'Rendimiento_Nivel'], observed=False).size().unstack(fill_value=0)

# Normalizar por filas para obtener probabilidades condicionales P(Rendimiento | Produccion, Area)
cpt_prob = cpt.div(cpt.sum(axis=1), axis=0).fillna(0)

print("📊 Tabla de Probabilidad Condicional P(Rendimiento_Nivel | Produccion_Nivel, Area_Nivel):")
cpt_prob.round(3)



## 5. Evaluación del Sistema y Matriz de Confusión

Para analizar en detalle el comportamiento del clasificador probabilístico en las distintas clases (`Alto`, `Medio`, `Bajo`), visualizaremos la **Matriz de Confusión**.

### Tarea 5.1: Matriz de Confusión Gráfica
Construya un mapa de calor (`heatmap`) que muestre los aciertos y desviaciones del modelo.



In [ ]:
# Generación de la Matriz de Confusión
labels_clase = ['Alto', 'Medio', 'Bajo']
cm = confusion_matrix(y_test, y_pred, labels=labels_clase)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=labels_clase, yticklabels=labels_clase)
plt.title('Matriz de Confusión - Clasificador Bayesiano')
plt.xlabel('Clase Predicha')
plt.ylabel('Clase Real (True)')
plt.grid(False)
plt.show()



## 6. Cuestionario de Evaluación y Trabajo Autónomo

Responda fundamentadamente a las siguientes preguntas en celdas de Markdown:

1. **Supuesto de Independencia de Naive Bayes:**  
   ¿Qué impacto tiene asumir que las variables de entrada son independientes cuando en la práctica existe una fuerte correlación entre el área cultivada y el volumen de producción? ¿En qué escenario una Red Bayesiana completa supera a un clasificador Naive Bayes?

2. **Diferencia entre MLE y MAP:**  
   Explique la diferencia teórica entre la Estimación por Máxima Verosimilitud (MLE - *Maximum Likelihood Estimation*) y la Estimación Máximo a Posteriori (MAP - *Maximum A Posteriori*).

3. **Análisis de Métricas en Clasificación:**  
   A partir del reporte de clasificación y la matriz de confusión generada en la Sección 5, identifique qué clase presentó la menor precisión y argumente a qué factor probabilístico o desbalance de datos se debe.

